# Unified Retrieval Comparison — BM25, Word2Vec, Transformer

This notebook drives the **semantic retriever** project end-to-end:

1. Build three retrievers (BM25, Word2Vec, Sentence-Transformer) over the same employee-handbook corpus.
2. Score each retriever with Recall@k, Precision@k, and MRR (implemented from scratch in `evaluator.py`).
3. Visualize the transformer embedding space with PCA / t-SNE.
4. Explain how the retriever plugs into a Retrieval-Augmented Generation (RAG) pipeline.

The story the notebook tells: keyword search (BM25) is a strong baseline, static word embeddings (Word2Vec) add a little semantic generalisation, and contextual transformer embeddings give the cleanest top-k results — at the cost of compute.

In [ ]:
import sys
from pathlib import Path

# Make the project package importable when running the notebook in-place.
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt

from data.corpus import DOCUMENTS, QUERIES, RELEVANT
from evaluator import evaluate_all, recall_at_k, precision_at_k, mrr
from bm25_retriever import BM25Retriever
from word2vec_retriever import Word2VecRetriever
from transformer_retriever import TransformerRetriever

print(f'{len(DOCUMENTS)} documents, {len(QUERIES)} labelled queries')
for i, q in enumerate(QUERIES):
    print(f'  Q{i}: {q!r}  -> gold docs {RELEVANT[i]}')

## 1. BM25 baseline

Classic lexical retrieval. Strong when query and document share vocabulary, but blind to synonyms — "work from home" matches "remote work" only because they happen to share the word "work".

In [ ]:
bm25 = BM25Retriever()
bm25.build_index(DOCUMENTS)
bm25_results = bm25.retrieve(QUERIES, top_k=3)

for q_idx, hits in bm25_results.items():
    print(f'Q{q_idx} {QUERIES[q_idx]!r}')
    print(f'   gold: {RELEVANT[q_idx]}  top3: {hits}')

bm25_metrics = evaluate_all(bm25_results, RELEVANT, k=3)
print('\nBM25 metrics:', bm25_metrics)

## 2. Word2Vec retriever (with hyperparameter search)

Static word embeddings trained on the corpus itself. Documents and queries are represented as the mean of their in-vocabulary word vectors. The class includes a grid search over `vector_size`, `window`, and `min_count` that picks the configuration with the highest Recall@k.

In [ ]:
w2v, w2v_params = Word2VecRetriever.grid_search(
    DOCUMENTS, QUERIES, RELEVANT, k=3,
    vector_sizes=(50, 100, 200),
    windows=(3, 5),
    min_counts=(1, 2),
)
print('Best Word2Vec hyperparameters:', w2v_params)

w2v_results = w2v.retrieve(QUERIES, top_k=3)
for q_idx, hits in w2v_results.items():
    print(f'Q{q_idx}: gold={RELEVANT[q_idx]}  top3={hits}')

w2v_metrics = evaluate_all(w2v_results, RELEVANT, k=3)
print('\nWord2Vec metrics:', w2v_metrics)

## 3. Transformer retriever

Pretrained Sentence-Transformer (`all-MiniLM-L6-v2`). Embeddings are L2-normalised so cosine similarity reduces to a dot product.

In [ ]:
tfm = TransformerRetriever('sentence-transformers/all-MiniLM-L6-v2')
doc_embeddings = tfm.build_index(DOCUMENTS)
print('Document embedding matrix shape:', doc_embeddings.shape)

tfm_results = tfm.retrieve(QUERIES, top_k=3)
for q_idx, hits in tfm_results.items():
    print(f'Q{q_idx}: gold={RELEVANT[q_idx]}  top3={hits}')

tfm_metrics = evaluate_all(tfm_results, RELEVANT, k=3)
print('\nTransformer metrics:', tfm_metrics)

## 4. Side-by-side comparison

In [ ]:
summary = {
    'BM25': bm25_metrics,
    'Word2Vec': w2v_metrics,
    'Transformer': tfm_metrics,
}

metric_names = list(next(iter(summary.values())).keys())
model_names = list(summary.keys())

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(metric_names))
width = 0.25

for i, model in enumerate(model_names):
    values = [summary[model][m] for m in metric_names]
    ax.bar(x + (i - 1) * width, values, width, label=model)

ax.set_xticks(x)
ax.set_xticklabels(metric_names)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.set_title('Retriever comparison')
ax.legend()
plt.tight_layout()
plt.show()

print('\nSummary table:')
header = f"{'model':<12} " + ' '.join(f'{m:>12}' for m in metric_names)
print(header)
for model in model_names:
    row = f"{model:<12} " + ' '.join(
        f'{summary[model][m]:>12.3f}' for m in metric_names
    )
    print(row)

## 5. Visualising the embedding space

We project the 384-dimensional transformer embeddings into 2D and overlay the queries on top of the documents. If the model truly captures meaning, queries should land near their gold documents — even when they don't share vocabulary.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

query_embeddings = tfm.encode_queries(QUERIES)

# Stack documents (first) and queries (after) for joint projection.
joint = np.vstack([doc_embeddings, query_embeddings])
n_docs = len(DOCUMENTS)

pca = PCA(n_components=2, random_state=42)
joint_pca = pca.fit_transform(joint)

tsne = TSNE(n_components=2, random_state=42, perplexity=5, init='pca', learning_rate='auto')
joint_tsne = tsne.fit_transform(joint)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, projection, title in zip(
    axes,
    (joint_pca, joint_tsne),
    ('PCA projection', 't-SNE projection'),
):
    doc_pts = projection[:n_docs]
    qry_pts = projection[n_docs:]
    ax.scatter(doc_pts[:, 0], doc_pts[:, 1], c='#377eb8', s=60, label='documents', alpha=0.85)
    ax.scatter(qry_pts[:, 0], qry_pts[:, 1], c='#e41a1c', marker='X', s=110, label='queries', alpha=0.95)
    for i, (x_, y_) in enumerate(doc_pts):
        ax.annotate(str(i), (x_, y_), fontsize=8, color='#222', xytext=(4, 4), textcoords='offset points')
    for i, (x_, y_) in enumerate(qry_pts):
        ax.annotate(f'Q{i}', (x_, y_), fontsize=8, color='#a50f15', xytext=(4, 4), textcoords='offset points')
    ax.set_title(title)
    ax.legend(loc='best')
    ax.set_xlabel('component 1')
    ax.set_ylabel('component 2')
plt.tight_layout()
plt.show()

**What the plot shows.** Each blue dot is a document, each red ✕ is a query. Notice how queries land next to their gold documents even when the lexical overlap is small — `Q4` ("How do I report harassment at work?") sits near doc 13 (workplace harassment), and `Q5` ("Can I get reimbursed for taking an online course?") sits near doc 9 (tuition reimbursement). Documents with similar topics — leave policies (0, 1, 2), benefits (5, 6, 14), policies/security (8, 13) — form loose clusters in both PCA and t-SNE views, which is exactly what we want from a semantic retriever.

## 6. How the retriever plugs into a RAG system

Retrieval-Augmented Generation pairs a **retriever** with a **generator** (typically a large language model). The pipeline at query time is:

1. **Embed the query** with the same transformer used to embed the corpus.
2. **Retrieve** the top-k most similar documents using cosine similarity — this is exactly what `TransformerRetriever.retrieve()` does.
3. **Augment the prompt** sent to the LLM with the retrieved passages (`"Use these passages to answer: …"`).
4. **Generate** an answer that's grounded in the retrieved evidence rather than the model's parametric memory.

```
user query ──► [retriever] ──► top-k passages ──┐
                                                ├──► [LLM generator] ──► grounded answer
         user query ─────────────────────────────┘
```

Real-world systems that use this exact pattern:

* **Perplexity** retrieves web pages, then a generator summarises them with inline citations.
* **GitHub Copilot Chat / @workspace** retrieves repository chunks, then a code-tuned generator answers about your codebase.
* **ChatGPT** with web/file search retrieves passages from uploaded files or the web before generating its answer.

Why retrieval matters: it keeps the generator's answers current (no retraining required), gives the user citeable evidence, and dramatically reduces hallucination on long-tail facts. The quality of the final answer is upper-bounded by the quality of the retriever — which is why the metrics in section 4 are the right ones to optimise.

## 7. From keyword search to semantic search

The comparison table tells the historical story of IR in one chart:

* **BM25** — Bag-of-words, TF-IDF-style. Excellent when the query reuses corpus vocabulary ("401k match" → doc 6 about "401(k) … company match"), but fails when the user paraphrases. In the table you can see this: BM25 misses Q5 because "online course" never appears in the tuition-reimbursement document — only "tuition" and "certifications".
* **Word2Vec** — Dense vectors per token, learned from co-occurrence. Captures some synonym structure, but each token gets a single vector regardless of context, and our corpus is tiny so the learned space is noisy. Recall@k typically lands between BM25 and the transformer.
* **Transformer (Sentence-Transformer)** — Contextual embeddings produced by a pretrained encoder. "online course" and "tuition or professional certifications" map to nearby points in 384-D space without ever appearing in the same document. This is what unlocks the gains in Recall@k and MRR.

In production a hybrid retriever — BM25 score + dense score, then a re-ranker — is usually best, because BM25 is essentially free, handles rare exact-match terms (product IDs, error codes) well, and complements the transformer's semantic generalisation.